# Clase 6 · ¿Son distintos entre sí?

**Estadística Descriptiva e Inferencial** · Módulo 2 · Sesión 6 de 14

Comparar dos grupos y comparar varios: la t de Welch, el ANOVA, las pruebas post-hoc
y las alternativas no paramétricas.

---

### Lo que este notebook hace distinto

No vas a aprender a llamar a `ttest_ind`. Eso son cinco minutos. Vas a hacer tres cosas
que casi nadie hace:

1. **Medir** la tasa real de error de la prueba que enseñan los libros, y verla fallar.
2. **Implementar** el ANOVA de Welch a mano, porque no está en scipy.
3. **Resolver** una contradicción real entre dos análisis defendibles del mismo dataset.

| Bloque | Tema | Min |
|---|---|---|
| 1 | Diseño y datos emparejados | 12 |
| 2 | Student vs Welch, por simulación | 16 |
| 3 | ANOVA y ANOVA de Welch | 14 |
| 4 | Post-hoc y la contradicción | 13 |
| 5 | Tamaño del efecto y no paramétricas | 10 |

> **SEED = 42.** El bloque 2 es el obligatorio.

## Celda 0 · Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEED = 42
rng = np.random.default_rng(SEED)

NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (9, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-6):
    if obtenido is None:
        print(f"[ ] {nombre}: todavia no calculaste nada (obtenido = None)")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}: obtenido = {float(obtenido):.6f} | "
          f"esperado = {float(esperado):.6f} (tolerancia {tol})")
    if not ok:
        print("      -> revisa este paso antes de continuar.")
    return ok

def check_bool(nombre, condicion, pista=""):
    print(f"{'[OK]' if condicion else '[X ]'} {nombre}")
    if not condicion and pista:
        print(f"      -> {pista}")
    return bool(condicion)

print("Entorno listo | SEED =", SEED)

### Tabla de símbolos → código

| Concepto | En el código |
|---|---|
| dos grupos independientes | `stats.ttest_ind(a, b, equal_var=False)` |
| dos medidas emparejadas | `stats.ttest_rel(antes, despues)` |
| igualdad de varianzas | `stats.levene(a, b)` |
| k grupos, varianzas iguales | `stats.f_oneway(g1, g2, g3)` |
| k grupos, varianzas desiguales | Welch-ANOVA (lo implementas en el bloque 3) |
| corrección por múltiples | `multipletests(ps, method='holm')` |

**El parámetro que hay que recordar:** `equal_var=False`. El default de scipy es `True`.

---
# Bloque 1 · Diseño y datos emparejados  ·  12 min

Primero construimos el dataset del curso: tiempo de resolución de casos de fraude por
canal de ingreso. Los tres canales tienen tamaños y dispersiones distintas — a propósito,
porque es el caso normal en cualquier operación real.

Después reproducimos el ejemplo de la slide 6, que es el que muestra por qué identificar
el diseño importa más que elegir bien la prueba.

In [ ]:
# ── DEMOSTRACIÓN: el dataset de los tres canales ─────────────────────────
g = np.random.default_rng(SEED)

app     = g.normal(12.0, 3.0, 60)   # canal digital: rapido y consistente
web     = g.normal(13.5, 3.2, 45)   # parecido a app
agencia = g.normal(17.0, 9.0, 25)   # mas lento y MUCHO mas variable, y con menos casos

canales = {"app": app, "web": web, "agencia": agencia}

resumen = pd.DataFrame({
    k: {"n": len(v), "media": v.mean(), "s": v.std(ddof=1), "varianza": v.var(ddof=1)}
    for k, v in canales.items()
}).T
print(resumen.round(3))
print()
print(f"La varianza de agencia es {agencia.var(ddof=1)/app.var(ddof=1):.1f} veces la de app.")
print("Y agencia es el grupo con MENOS casos. Esa combinacion -- grupo chico y disperso --")
print("es justo la que rompe la prueba de Student, como veras en el bloque 2.")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.boxplot([app, web, agencia], labels=["app", "web", "agencia"], vert=False,
           patch_artist=True,
           boxprops=dict(facecolor="#C9D8F5", edgecolor=NAVY),
           medianprops=dict(color=MAG, linewidth=2))
ax.set_xlabel("tiempo de resolución (días)")
ax.set_title("Los tres canales", color=NAVY, fontweight="bold")
plt.tight_layout(); plt.show()

### Ejercicio 1.1 — El diagnóstico de varianzas

Antes de comparar nada, mira si las dispersiones se parecen. La prueba de Levene contrasta
H₀: «todas las varianzas son iguales».

**Ojo con la lógica:** Levene rechazando significa que las varianzas SÍ difieren, y eso
te dice que no puedes usar las pruebas que asumen igualdad.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
p_levene_2 = None    # Levene sobre app y agencia
p_levene_3 = None    # Levene sobre los tres canales

print(f"Levene app-agencia: p = {p_levene_2}")
print(f"Levene los tres   : p = {p_levene_3}")

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check_bool("Levene rechaza para app vs agencia (p < 0.01)", p_levene_2 < 0.01),
     check_bool("Levene rechaza para los tres canales (p < 0.01)", p_levene_3 < 0.01)]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

### Ejercicio 1.2 — El costo de confundir el diseño

Ahora el ejemplo de la slide 6: **los mismos 30 analistas** medidos antes y después de una
capacitación. El efecto real que introducimos es de 2.5 días de mejora.

Analízalo de las dos formas y compara los p-valores.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
g = np.random.default_rng(SEED)
antes   = g.normal(20, 10, 30)             # los analistas son MUY distintos entre si
despues = antes - 2.5 + g.normal(0, 2, 30) # cada uno mejora ~2.5 dias

p_emparejada    = None   # la prueba CORRECTA para este diseño
p_independiente = None   # la prueba incorrecta, como si fueran dos grupos distintos

print(f"emparejada: {p_emparejada} | independiente: {p_independiente}")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check_bool("la emparejada detecta el efecto con contundencia (p < 1e-5)",
                p_emparejada < 1e-5),
     check_bool("la independiente NO lo detecta (p > 0.05)",
                p_independiente > 0.05,
                "si te da significativo, revisa que uses ttest_ind y no ttest_rel"),
     check_bool("la diferencia entre ambos p-valores es de varios órdenes de magnitud",
                p_independiente / p_emparejada > 1e6)]
print()
print("Los datos son IDENTICOS en los dos analisis. Lo unico que cambio fue la prueba.")
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.2")

---
# Bloque 2 · Student vs Welch, medido  ·  16 min

**Este es el bloque obligatorio.**

La slide 8 afirmó que Student rechaza el 17 % de las veces cuando debería rechazar el 5 %.
Vamos a medirlo en lugar de creerlo.

El método es el mismo del bloque 3 de la Clase 5: simular un mundo donde **H₀ es
verdadera** y contar cuántas veces la prueba la rechaza.

### Ejercicio 2.1 — Las dos pruebas sobre nuestros datos

Antes de simular, corre las dos sobre app vs agencia y compara.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
res_student = None   # ttest_ind con equal_var=True
res_welch   = None   # ttest_ind con equal_var=False

print("Student:", res_student)
print("Welch  :", res_welch)

In [ ]:
# ── VERIFICACIÓN 2.1 ─────────────────────────────────────────────────────
r = [check("t de Student", res_student.statistic, -3.6423, tol=1e-3),
     check("p de Student", res_student.pvalue, 0.000469, tol=1e-5),
     check("t de Welch", res_welch.statistic, -2.5235, tol=1e-3),
     check("p de Welch", res_welch.pvalue, 0.018132, tol=1e-5),
     check("gl de Welch", res_welch.df, 25.76, tol=0.05)]
print()
print("2.1 OK" if all(r) else "Revisa 2.1")

### Ejercicio 2.2 — La simulación que decide

Escribe una función que, para un escenario dado (tamaños y desviaciones), simule muchas
parejas de muestras **con la misma media** y devuelva qué porcentaje de veces cada prueba
rechaza H₀.

**Pista:** `stats.ttest_ind` acepta arrays 2D con `axis=1`, así que puedes hacer las
20 000 pruebas de un golpe sin bucle.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
def tasa_error_tipo_I(n1, n2, sd1, sd2, reps=20_000, semilla=7):
    """Devuelve (% rechazos de Student, % rechazos de Welch) cuando H0 es VERDADERA."""
    g = np.random.default_rng(semilla)
    # Las dos poblaciones tienen media 0: no hay ninguna diferencia real.
    a = g.normal(0, sd1, size=(reps, n1))
    b = g.normal(0, sd2, size=(reps, n2))
    # TU CÓDIGO: corre las dos pruebas con axis=1 y cuenta p < 0.05
    return None, None

print(tasa_error_tipo_I(60, 25, 3, 9))

Ahora los cuatro escenarios de la slide 8.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
escenarios = [
    (60, 25, 3, 9, "el grupo chico es el disperso"),
    (25, 60, 3, 9, "el grupo grande es el disperso"),
    (30, 30, 3, 9, "n iguales, varianzas distintas"),
    (50, 50, 3, 3, "supuestos cumplidos"),
]

filas = []
for n1, n2, s1, s2, nota in escenarios:
    st, we = None, None
    filas.append({"escenario": f"n=({n1},{n2}) sd=({s1},{s2})",
                  "Student_%": st, "Welch_%": we, "nota": nota})

tabla_tipo_I = pd.DataFrame(filas)
print(tabla_tipo_I)

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
tabla_tipo_I = pd.DataFrame(filas)
r = [check_bool("escenario 1: Student se pasa del 12 %",
                tabla_tipo_I.loc[0, "Student_%"] > 12,
                "debería salir alrededor de 17 %"),
     check_bool("escenario 2: Student baja del 2 % (ultraconservador)",
                tabla_tipo_I.loc[1, "Student_%"] < 2),
     check_bool("Welch se mantiene entre 4 % y 6 % en los CUATRO escenarios",
                ((tabla_tipo_I["Welch_%"] > 4) & (tabla_tipo_I["Welch_%"] < 6)).all()),
     check_bool("escenario 4: las dos coinciden (supuestos cumplidos)",
                abs(tabla_tipo_I.loc[3, "Student_%"] - tabla_tipo_I.loc[3, "Welch_%"]) < 0.5)]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.2")

---
# Bloque 3 · ANOVA y ANOVA de Welch  ·  14 min

`scipy` tiene `f_oneway` (el ANOVA clásico), pero **no tiene el ANOVA de Welch**. Y como
nuestras varianzas son desiguales, el clásico no sirve.

Así que lo implementas tú. Son unas quince líneas y entender la fórmula vale la pena,
porque es exactamente la misma idea de la t de Welch generalizada a k grupos.

In [ ]:
# ── DEMOSTRACIÓN: el ANOVA clásico y su problema ─────────────────────────
f_cls = stats.f_oneway(app, web, agencia)
kw    = stats.kruskal(app, web, agencia)

print(f"ANOVA clasico   : F = {f_cls.statistic:.4f}   p = {f_cls.pvalue:.6f}")
print(f"Kruskal-Wallis  : H = {kw.statistic:.4f}   p = {kw.pvalue:.6f}")
print(f"Levene          : p = {stats.levene(app, web, agencia).pvalue:.3e}")
print()
print("El ANOVA clasico da un p muy chico. Pero Levene dice que sus supuestos no se")
print("cumplen, asi que ese p no es de fiar. Y Kruskal, que no asume varianzas iguales,")
print("da un p treinta veces mas grande. Esa discrepancia es la senal de alarma.")

### Ejercicio 3.1 — Implementa el ANOVA de Welch

La fórmula, paso a paso. Para k grupos con tamaños nᵢ, medias mᵢ y varianzas vᵢ:

1. Pesos: `w_i = n_i / v_i`
2. Media ponderada: `m_w = Σ(w_i · m_i) / Σw_i`
3. Numerador: `A = Σ(w_i · (m_i − m_w)²) / (k − 1)`
4. Término de corrección: `lam = Σ[ (1 − w_i/Σw)² / (n_i − 1) ]`
5. Denominador: `B = 1 + [2(k−2) / (k²−1)] · lam`
6. Estadístico: `F = A / B`
7. Grados de libertad: `df1 = k − 1`, `df2 = (k²−1) / (3·lam)`
8. p-valor: `stats.f.sf(F, df1, df2)`

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
def welch_anova(grupos):
    """ANOVA de Welch para k grupos con varianzas desiguales.
    Devuelve (F, df1, df2, p)."""
    k = len(grupos)
    n = np.array([len(x) for x in grupos])
    m = np.array([x.mean() for x in grupos])
    v = np.array([x.var(ddof=1) for x in grupos])

    # TU CÓDIGO: sigue los ocho pasos del enunciado
    return None, None, None, None

print(welch_anova([app, web, agencia]))

In [ ]:
# ── VERIFICACIÓN 3.1 ─────────────────────────────────────────────────────
Fw, d1, d2, pw = welch_anova([app, web, agencia])
r = [check("F de Welch-ANOVA", Fw, 3.8278, tol=1e-3),
     check("gl del denominador", d2, 52.25, tol=0.05),
     check("p de Welch-ANOVA", pw, 0.028098, tol=1e-5),
     check_bool("y el p del clásico es mucho más pequeño", f_cls.pvalue < pw / 10)]
print()
print("3.1 OK" if all(r) else "Revisa 3.1 — el paso que más se equivoca es el 4 (lam)")

### Ejercicio 3.2 — ¿Controla el error donde el clásico no?

Mide la tasa de error tipo I de las dos versiones, igual que en el bloque 2 pero con
tres grupos.

Con 3 000 réplicas basta: el ANOVA de Welch no está vectorizado y hay que hacer bucle.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
def tasa_anova(ns, sds, reps=3000, semilla=11):
    """% de rechazos del ANOVA clásico y del de Welch cuando H0 es VERDADERA."""
    g = np.random.default_rng(semilla)
    # Los k grupos tienen todos media 0.
    return None, None

for ns, sds in [((60, 45, 25), (3, 3.2, 9)), ((25, 45, 60), (9, 3.2, 3))]:
    print(ns, sds, tasa_anova(ns, sds))

In [ ]:
# ── VERIFICACIÓN 3.2 ─────────────────────────────────────────────────────
tabla_anova = pd.DataFrame(filas_a)
r = [check_bool("el ANOVA clásico se pasa del 12 % en los dos primeros escenarios",
                (tabla_anova.loc[0, "ANOVA_clasico_%"] > 12) and
                (tabla_anova.loc[1, "ANOVA_clasico_%"] > 12)),
     check_bool("el de Welch se mantiene entre 4 % y 6.5 % en los tres",
                ((tabla_anova["ANOVA_Welch_%"] > 4) &
                 (tabla_anova["ANOVA_Welch_%"] < 6.5)).all())]
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa 3.2")

---
# Bloque 4 · Post-hoc y la contradicción  ·  13 min

El ANOVA dijo «hay alguna diferencia». Ahora hay que averiguar **cuál**, y aquí aparece
el conflicto de la slide 14.

Dos caminos, los dos estándar:
- **Tukey HSD**, la prueba post-hoc clásica. Asume varianzas iguales.
- **t de Welch por pares + corrección de Holm.** No asume nada de eso.

Los vas a correr los dos y no van a coincidir.

### Ejercicio 4.1 — Tukey HSD

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# pairwise_tukeyhsd necesita los datos APILADOS en un solo array,
# más un array de etiquetas de grupo del mismo largo.
valores   = None
etiquetas = None

tukey = None
print(tukey)

In [ ]:
# ── VERIFICACIÓN 4.1 ─────────────────────────────────────────────────────
res_t = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
n_sig_tukey = sum(bool(v) for v in res_t["reject"])
r = [check_bool("Tukey encuentra 2 comparaciones significativas de 3", n_sig_tukey == 2),
     check_bool("y la de app vs web NO es significativa",
                not bool(res_t[(res_t["group1"] == "app") & (res_t["group2"] == "web")]["reject"].iloc[0]))]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

### Ejercicio 4.2 — Welch por pares + Holm

Ahora el otro camino: una t de Welch para cada par y después corregir los tres p-valores
con Holm, como en la Clase 5.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
from statsmodels.stats.multitest import multipletests
from itertools import combinations

pares = list(combinations(canales.keys(), 2))

ps_crudos = None    # lista con el p de Welch de cada par
ps_holm   = None    # los mismos p, ajustados con Holm

tabla_ph = None     # DataFrame con par, diferencia, p crudo y p ajustado
print(tabla_ph)

In [ ]:
# ── VERIFICACIÓN 4.2 ─────────────────────────────────────────────────────
r = [check("p de Welch para app vs agencia", ps_crudos[1], 0.018132, tol=1e-4),
     check("p ajustado con Holm para app vs agencia", ps_holm[1], 0.054397, tol=1e-4),
     check_bool("ninguna comparación sobrevive Holm", rechaza.sum() == 0),
     check_bool("y Tukey sí encontraba dos: hay contradicción",
                n_sig_tukey == 2 and rechaza.sum() == 0)]
print()
print("4.2 OK" if all(r) else "Revisa 4.2")

### Ejercicio 4.3 — Resuélvelo tú

Este ejercicio **no tiene una única respuesta numérica**. Se te pide escribir una
conclusión razonada, que es lo que se te va a pedir en el trabajo.

Ten en cuenta lo que ya sabes:
- Levene rechaza igualdad de varianzas (bloque 1), y Tukey la asume.
- La diferencia app-agencia es de 4.1 días, con d = 0.87 (efecto grande).
- El p ajustado más pequeño es 0.054.
- Agencia tiene solo 25 casos.

Completa las tres variables y después compara con la solución.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# 1. ¿Qué método es el defendible aquí, y por qué?
metodo_elegido = ""      # "tukey" o "welch_holm"
razon          = ""      # una frase

# 2. ¿Cuál es la conclusión honesta sobre si agencia es más lenta?
conclusion = """
(escribe aquí tu conclusión en dos o tres frases)
"""

# 3. ¿Qué recomendarías hacer a continuación?
recomendacion = ""

print(metodo_elegido, "|", razon)
print(conclusion)
print(recomendacion)

In [ ]:
# ── VERIFICACIÓN 4.3 ─────────────────────────────────────────────────────
r = [check_bool("elegiste welch_holm", metodo_elegido == "welch_holm",
                "Levene rechaza igualdad de varianzas, así que Tukey no es defendible aquí"),
     check_bool("escribiste una razón", len(razon) > 20),
     check_bool("escribiste una conclusión", len(conclusion.strip()) > 40),
     check_bool("escribiste una recomendación", len(recomendacion) > 20)]
print()
print("Bloque 4 COMPLETO" if all(r) else "Completa las tres respuestas de 4.3")

---
# Bloque 5 · Tamaño del efecto y no paramétricas  ·  10 min

Cerramos con lo que el p-valor no dice: cuánto. Y con la comprobación de que un enfoque
completamente distinto llega a la misma conclusión.

### Ejercicio 5.1 — d de Cohen, η² y ω²

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
def d_cohen(a, b):
    """d de Cohen con desviación estándar agrupada."""
    return None

# Y ahora el efecto global del ANOVA
valores_all = np.concatenate([app, web, agencia])
k, N = 3, len(valores_all)

eta2   = None   # SS_entre / SS_total
omega2 = None   # (SS_entre − (k−1)·MS_dentro) / (SS_total + MS_dentro)

print(f"d(app, agencia) = {d_cohen(app, agencia)}")
print(f"eta2 = {eta2} | omega2 = {omega2}")

In [ ]:
# ── VERIFICACIÓN 5.1 ─────────────────────────────────────────────────────
r = [check("d de Cohen para app vs agencia", d_cohen(app, agencia), -0.8670, tol=1e-3),
     check("eta cuadrado", eta2, 0.1259, tol=1e-3),
     check("omega cuadrado", omega2, 0.1114, tol=1e-3),
     check_bool("omega² es menor que eta² (la corrección va hacia abajo)", omega2 < eta2)]
print()
print("5.1 OK" if all(r) else "Revisa 5.1")

### Ejercicio 5.2 — ¿Coinciden las no paramétricas?

Si un enfoque que no asume normalidad ni varianzas iguales llega a la misma conclusión,
tu resultado es más creíble. Compruébalo.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
p_mw       = None   # Mann-Whitney para app vs agencia
p_kruskal  = None   # Kruskal-Wallis para los tres
p_wilcoxon = None   # Wilcoxon para antes/despues del bloque 1

print(f"Mann-Whitney: {p_mw} | Kruskal: {p_kruskal} | Wilcoxon: {p_wilcoxon}")

In [ ]:
# ── VERIFICACIÓN 5.2 ─────────────────────────────────────────────────────
r = [check("p de Mann-Whitney", p_mw, 0.010446, tol=1e-4),
     check("p de Kruskal-Wallis", p_kruskal, 0.018199, tol=1e-4),
     check_bool("Wilcoxon detecta el efecto emparejado con contundencia",
                p_wilcoxon < 1e-4),
     check_bool("Welch y Mann-Whitney coinciden en la conclusión a α = 0.05",
                (res_welch.pvalue < 0.05) == (p_mw < 0.05))]
print()
print("Bloque 5 COMPLETO — laboratorio terminado" if all(r) else "Revisa 5.2")

---
# Cierre

### Checklist de salida

- [ ] Identifico si el diseño es emparejado o independiente antes de elegir la prueba.
- [ ] Uso `equal_var=False` y sé explicar por qué con un número.
- [ ] Medí la tasa de error de Student y la vi llegar al 17 %.
- [ ] Implementé el ANOVA de Welch y sé por qué hace falta.
- [ ] Sé hacer post-hoc corrigiendo por comparaciones múltiples.
- [ ] Resolví una contradicción entre dos análisis usando los supuestos como criterio.
- [ ] Reporto d o η² además del p-valor.

### Lo que quedó demostrado con números

| Bloque | Lo que viste |
|---|---|
| 1 | La prueba correcta da p = 4 × 10⁻⁷; la incorrecta, p = 0.13. Mismos datos. |
| 2 | Student rechaza el 17 % de las veces cuando promete 5 %. Welch se queda en 5 %. |
| 3 | El ANOVA clásico repite el mismo fallo; el de Welch lo corrige. |
| 4 | Tukey encuentra dos diferencias, Welch+Holm ninguna. Los supuestos deciden. |
| 5 | El canal explica el 12.6 % de la variación: real, pero no es la palanca principal. |

El bloque 2 es el que cambia tu práctica: una línea de código. El bloque 4 es el que
cambia tu criterio, y es más difícil.

### Reto para la próxima clase

Busca una comparación de grupos que exista en tu trabajo: canales, sucursales, segmentos,
cohortes, antes y después de un cambio. Responde cuatro cosas:

1. ¿El diseño es emparejado o independiente?
2. ¿Las dispersiones de los grupos se parecen?
3. ¿Cuántas comparaciones hay en total?
4. ¿Qué prueba se usó, y quién decidió que fuera esa?

### Clase 7

**Variables categóricas:** la prueba χ² de independencia, las tablas de contingencia y
las medidas de asociación. Pasamos de comparar promedios a comparar proporciones —
aprobado o rechazado, alerta o no alerta, incumple o no incumple.

---
*Estadística Descriptiva e Inferencial · Módulo 2 · Clase 6 · SEED = 42*